# 🏷️ Lesson 05: Metadata

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/05_metadata.ipynb)

> **The text tells the model *what*. The metadata tells *us* where it came from.**

---

## 🎯 Goal
By the end of this lesson you can attach useful metadata to every `Document`, filter documents by it, and build a source citation from it.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~15 minutes |
| 📚 **Prerequisites** | [Lesson 04](04_cleaning.ipynb) |
| 🔑 **API key** | Not needed |
| 🛠 **Libraries** | `pdfplumber`, `langchain-core` |

In [ ]:
!pip install -q -U pdfplumber langchain-core

In [1]:
from pathlib import Path
import urllib.request
import re
import unicodedata
from collections import Counter
import pdfplumber
from langchain_core.documents import Document

PDF_URL = "https://raw.githubusercontent.com/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/main/data/pdf1.pdf"

pdf_path = next((p for p in [Path("data/pdf1.pdf"), Path("../data/pdf1.pdf")] if p.exists()), None)
if pdf_path is None:
    pdf_path = Path("data/pdf1.pdf")
    pdf_path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PDF_URL, pdf_path)

# Lesson 03: parse into sections
lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages):
        for line in page.extract_text_lines():
            size = round(max(char["size"] for char in line["chars"]), 1)
            lines.append({"text": line["text"], "size": size, "page": page_number})

body_size = Counter(line["size"] for line in lines).most_common(1)[0][0]

sections = []
current = None
for line in lines:
    if line["size"] > body_size:
        current = {"title": line["text"], "page": line["page"], "text": []}
        sections.append(current)
    elif current is not None:
        current["text"].append(line["text"])
sections = [section for section in sections if section["text"]]

# Lesson 04: clean
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("(cid:127)", "\n•")
    text = re.sub(r"Full[n■]time", "Full-time", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

cleaned_documents = [
    Document(
        page_content=clean_text(" ".join(section["text"])),
        metadata={"source": pdf_path.name, "page": section["page"], "section": section["title"]},
    )
    for section in sections
]

print("documents:", len(cleaned_documents))

documents: 4


## 🏷️ What is metadata, and why do we need it?

**Metadata is information *about* the text, not the text itself.** We already have some: `source`, `page`, `section`. It is useful for three things:

| Use | Example |
|---|---|
| **Filtering** | "search only inside the *Paid Time Off* section" |
| **Citation** | "Answer taken from *pdf1.pdf*, section *Travel & Expense Policy*" |
| **Debugging** | "why did the system retrieve *this* chunk?" |

Metadata comes from three places:
1. **From the file:** name, page number.
2. **From the structure:** the section title (we got it in Lesson 03).
3. **Computed by us:** length, word count, a unique id, simple flags.

Let's add the third kind.

In [2]:
enriched_documents = []
for index, doc in enumerate(cleaned_documents):
    metadata = dict(doc.metadata)
    metadata.update({
        "doc_id": f"{pdf_path.stem}-{index}",
        "char_count": len(doc.page_content),
        "word_count": len(doc.page_content.split()),
        "has_list": "•" in doc.page_content,
    })
    enriched_documents.append(Document(page_content=doc.page_content, metadata=metadata))

for doc in enriched_documents:
    print(doc.metadata)

{'source': 'pdf1.pdf', 'page': 0, 'section': 'Non-Disclosure Agreement (NDA) Policy', 'doc_id': 'pdf1-0', 'char_count': 427, 'word_count': 53, 'has_list': False}
{'source': 'pdf1.pdf', 'page': 0, 'section': 'Workplace Conduct Policy', 'doc_id': 'pdf1-1', 'char_count': 360, 'word_count': 43, 'has_list': False}
{'source': 'pdf1.pdf', 'page': 0, 'section': 'Paid Time Off (PTO) Policy', 'doc_id': 'pdf1-2', 'char_count': 558, 'word_count': 96, 'has_list': True}
{'source': 'pdf1.pdf', 'page': 0, 'section': 'Travel & Expense Policy', 'doc_id': 'pdf1-3', 'char_count': 462, 'word_count': 59, 'has_list': False}


## 🔎 Use 1: Filtering

Metadata lets us select documents **before** doing any expensive search. Later, a vector database will do this for us. For now, plain Python is enough to see the idea.

In [3]:
def filter_documents(documents, **conditions):
    return [
        doc for doc in documents
        if all(doc.metadata.get(key) == value for key, value in conditions.items())
    ]

# exact match on one field
list_documents = filter_documents(enriched_documents, has_list=True)
print("has a list:", [doc.metadata["section"] for doc in list_documents])

# any condition you can write in Python
long_documents = [doc for doc in enriched_documents if doc.metadata["word_count"] > 55]
print("more than 55 words:", [doc.metadata["section"] for doc in long_documents])

# no match returns an empty list, not an error
print("page 5:", filter_documents(enriched_documents, page=5))

has a list: ['Paid Time Off (PTO) Policy']
more than 55 words: ['Paid Time Off (PTO) Policy', 'Travel & Expense Policy']
page 5: []


## 📎 Use 2: Citations

A good RAG answer says **where it came from**, so the user can check it. We build the citation from the metadata of the chunk we retrieved.

One detail: our `page` starts at **0** (as in Python), but a human reads "page 1" for the first page. So we show `page + 1`.

In [4]:
def make_citation(doc):
    metadata = doc.metadata
    return f'{metadata["source"]}, section "{metadata["section"]}", page {metadata["page"] + 1}'

for doc in enriched_documents:
    print("-", make_citation(doc))

- pdf1.pdf, section "Non-Disclosure Agreement (NDA) Policy", page 1
- pdf1.pdf, section "Workplace Conduct Policy", page 1
- pdf1.pdf, section "Paid Time Off (PTO) Policy", page 1
- pdf1.pdf, section "Travel & Expense Policy", page 1


## 📌 Summary

- **Metadata** is information *about* the text. It is used for **filtering**, **citations** and **debugging**.
- It comes from three places: the **file** (source, page), the **structure** (section title) and things we **compute** (id, length, flags).
- A filter can be a simple exact match, or any condition you can write in Python.
- A citation is built from the metadata of the retrieved chunk. Remember that our `page` starts at **0**, so we show `page + 1` to humans.
- Metadata is attached **now**, while we still know where everything came from. Once the text is split into chunks, this information is very hard to recover.

## ⚠️ Things to watch out for

- **Keep values simple.** Use text, numbers and `True`/`False`. Some vector databases only accept simple values (not lists or nested dictionaries), so check the one you plan to use.
- **Don't add metadata "just in case".** Every field should have a use: a filter you will really run, or something you will show the user.
- **Metadata can be wrong too.** Our `section` came from a font-size guess in Lesson 03. If that guess is wrong, every citation built on it is wrong.
- **Only one page here.** All our documents say `page: 0`, so page-based filtering isn't very interesting yet. It will be with a longer PDF.

## ✍️ Exercise (10 minutes)

1. Add a new metadata field `mentions_days` that is `True` when the text contains the word `"days"`.
2. Use `filter_documents` to find all documents where `mentions_days` is `True`.
3. Print their citations with `make_citation`.

**Success:** you can list the sections that mention "days", with a citation for each.

💡 Solution: `solutions/05_metadata.ipynb` (try on your own first)

## 🔥 Challenge (optional)

The PDF file itself has metadata (creation date, producer). With `pdfplumber` it is available as `pdf.metadata`.

1. Print it for our PDF. Is it useful? (Hint: look at the author field.)
2. The creation date looks like `D:20251120232316+00'00'`. Convert it to a readable date like `2025-11-20` and add it as `created` to every document.

Think about it: when would a "created" date matter in a real RAG system? (Hint: two versions of the same policy.)

## 📖 Next

Our documents are clean, organized and labeled. But a section can still be too long to send to a model as one piece. In the next lesson we split the text into **chunks**.

**➡️ Next lesson:** [06 Chunking](06_chunking.ipynb)